# SwiGLU-5: efficient global-recovery search and confirmation

This load-only report separates imported SwiGLU-3 evidence, the reconfigured `S5-C0` legacy control, new SwiGLU-5 challengers, and any later confirmation artifact. It does not load a model or dataset.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


artifact loading

In [ ]:
SEARCH_ARTIFACT = Path(
    'data/results/workflows/model/swiglu-5/search/swiglu-5-search-<timestamp>.json'
)
CONFIRMATION_ARTIFACTS = {
    # '0.2': Path('data/results/workflows/model/swiglu-5/confirmation/<artifact>.json'),
    # '0.5': Path('data/results/workflows/model/swiglu-5/confirmation/<artifact>.json'),
}

search = json.loads(SEARCH_ARTIFACT.read_text(encoding='utf-8'))
if search.get('workflow') != 'swiglu-5-search':
    raise ValueError('Expected a swiglu-5-search artifact')
if search.get('status') != 'completed':
    raise ValueError(f"Search is not completed: {search.get('status')}")
confirmations = {
    target: json.loads(path.read_text(encoding='utf-8'))
    for target, path in CONFIRMATION_ARTIFACTS.items()
}
for target, artifact in confirmations.items():
    if artifact.get('workflow') != 'swiglu-5-confirmation':
        raise ValueError(f'Expected a confirmation artifact for {target}')
    if artifact.get('status') != 'completed':
        raise ValueError(f'Confirmation {target} is not completed')
    if str(artifact['configuration']['target']) != str(float(target)):
        raise ValueError(f'Confirmation artifact target differs for {target}')

pd.DataFrame([
    {'evidence': 'SwiGLU-3', 'role': 'imported published baseline',
     'artifact': search['provenance']['swiglu_3']['path']},
    {'evidence': 'S5-C0', 'role': 'reproduced operators; reconfigured recovery',
     'artifact': str(SEARCH_ARTIFACT)},
    {'evidence': 'S5-C1..C4', 'role': 'new SwiGLU-5 challengers',
     'artifact': str(SEARCH_ARTIFACT)},
    {'evidence': 'confirmation',
     'role': 'executed 10M-100M continuation' if confirmations else 'planned; not supplied',
     'artifact': ', '.join(map(str, CONFIRMATION_ARTIFACTS.values())) or None},
])


SwiGLU-3 and dense references

In [ ]:
baseline_rows = []
dense = search['results']['dense_baseline']
baseline_rows.append({
    'target': 'dense', 'source': 'SwiGLU-5 fixed evaluation', 'tokens': 0,
    'validation_kl_t1': dense['recovery_validation_kl'],
    'wikitext_ppl': dense['wikitext_validation']['perplexity'],
})
for target, evidence in search['results']['published_swiglu_3'].items():
    pre = evidence['pre_recovery']
    baseline_rows.append({
        'target': target, 'source': 'published SwiGLU-3 pre-recovery', 'tokens': 0,
        'validation_kl_t1': None,
        'wikitext_ppl': pre['wikitext_validation']['perplexity'],
    })
    for milestone in evidence['milestones']:
        current = milestone['current']
        baseline_rows.append({
            'target': target, 'source': 'published SwiGLU-3 current',
            'tokens': milestone['actual_tokens'],
            'validation_kl_t1': current['recovery_validation_kl'],
            'wikitext_ppl': current['wikitext_validation']['perplexity'],
        })
baseline_df = pd.DataFrame(baseline_rows)
baseline_df


candidate comparison and allocations

In [ ]:
candidate_rows = []
allocation_rows = []
for target, candidates in search['results']['candidates'].items():
    for candidate_id, candidate in candidates.items():
        pre = candidate['pre_recovery']
        candidate_rows.append({
            'target': target, 'candidate': candidate_id,
            'initialization': candidate['initialization'],
            'allocation_method': candidate['allocation_method'],
            'realized_removal': candidate['realized_eligible_mlp_removal'],
            'pre_recovery_kl': pre['recovery_validation_kl'],
            'pre_recovery_ppl': pre['wikitext_validation']['perplexity'],
            'recovery_status': candidate['recovery']['status'],
        })
        for row in candidate['allocation']:
            allocation_rows.append({
                'target': target, 'candidate': candidate_id,
                'layer': row['layer'], 'width': row['replacement_width'],
                'dense_retained': row['retains_dense_module'],
                'output_bias': row['has_output_bias'],
            })
candidate_df = pd.DataFrame(candidate_rows).sort_values(['target', 'candidate'])
control_pre = candidate_df[candidate_df['candidate'] == 'S5-C0'].set_index('target')
candidate_df['pre_kl_minus_c0'] = candidate_df.apply(
    lambda row: row['pre_recovery_kl'] - control_pre.loc[row['target'], 'pre_recovery_kl'], axis=1
)
candidate_df['pre_ppl_minus_c0'] = candidate_df.apply(
    lambda row: row['pre_recovery_ppl'] - control_pre.loc[row['target'], 'pre_recovery_ppl'], axis=1
)
allocation_df = pd.DataFrame(allocation_rows).sort_values(['target', 'candidate', 'layer'])
display(candidate_df)
display(allocation_df.pivot_table(index=['target', 'layer'], columns='candidate', values='width'))


In [ ]:
for target, frame in allocation_df.groupby('target'):
    pivot = frame.pivot(index='layer', columns='candidate', values='width')
    ax = pivot.plot(figsize=(11, 4), marker='o', title=f'{target} removal: retained widths')
    ax.set_ylabel('SwiGLU width')
    ax.grid(alpha=0.25)
    plt.show()


recovery curves, dense gaps, and selection

In [ ]:
curve_rows = []
search_curve_limit = max(
    candidate['recovery'].get('tokens_seen', 0)
    for candidates in search['results']['candidates'].values()
    for candidate in candidates.values()
)
for target, evidence in search['results']['published_swiglu_3'].items():
    for row in evidence['validation_history']:
        if row['tokens_seen'] > search_curve_limit:
            continue
        curve_rows.append({
            'target': target, 'candidate': 'published-swiglu-3',
            'tokens': row['tokens_seen'],
            'validation_kl_t1': row['recovery_validation_kl'],
        })
for target, candidates in search['results']['candidates'].items():
    for candidate_id, candidate in candidates.items():
        for row in candidate['recovery']['validation_history']:
            curve_rows.append({
                'target': target, 'candidate': candidate_id,
                'tokens': row['actual_tokens'],
                'validation_kl_t1': row['recovery_validation_kl'],
            })
curve_df = pd.DataFrame(curve_rows)
for target, frame in curve_df.groupby('target'):
    fig, ax = plt.subplots(figsize=(10, 4))
    for candidate_id, candidate_frame in frame.groupby('candidate'):
        candidate_frame.sort_values('tokens').plot(
            x='tokens', y='validation_kl_t1', marker='o',
            label=candidate_id, ax=ax,
        )
    ax.set_title(f'{target} removal: fixed T=1 recovery KL')
    ax.grid(alpha=0.25)
    plt.show()


In [ ]:
selection_rows = []
guardrail_rows = []
for target in ('0.2', '0.5'):
    selected = search['results']['selection'][target]
    selection_rows.append({
        'target': target, 'winner': selected['winner_candidate_id'],
        **{f"dense_gap_{key}": value for key, value in selected['gaps']['to_dense'].items()},
        **{f"control_gap_{key}": value for key, value in selected['gaps']['to_s5_c0'].items()},
    })
    for row in selected['guardrail_decisions']:
        guardrail_rows.append({'target': target, **row})
display(pd.DataFrame(selection_rows))
display(pd.DataFrame(guardrail_rows).sort_values(['target', 'candidate_id']))
display(pd.DataFrame(search['results']['runtime']))
recovery_runtime_rows = []
for target, candidates in search['results']['candidates'].items():
    for candidate_id, candidate in candidates.items():
        recovery = candidate['recovery']
        recovery_runtime_rows.append({
            'target': target, 'candidate': candidate_id,
            'tokens': recovery.get('tokens_seen', 0),
            'training_seconds': recovery.get('training_seconds', 0.0),
            'evaluation_seconds': recovery.get('evaluation_seconds', 0.0),
            'checkpoint_seconds': recovery.get('checkpoint_seconds', 0.0),
        })
display(pd.DataFrame(recovery_runtime_rows).sort_values(['target', 'candidate']))
display(pd.DataFrame([search['results']['storage_preflight']]))
display(pd.DataFrame([search['results']['runtime_guard']]))


optional confirmation comparisons

In [ ]:
confirmation_rows = []
for target, artifact in confirmations.items():
    for row in artifact['results']['paired_swiglu_3_comparison']:
        confirmation_rows.append({
            'target': target, 'requested_tokens': row['requested_tokens'],
            **row['difference_swiglu_5_minus_swiglu_3'],
        })
if confirmation_rows:
    display(pd.DataFrame(confirmation_rows).sort_values(['target', 'requested_tokens']))
else:
    display(pd.DataFrame([{'status': 'Confirmation planned but not supplied'}]))
